In [1]:
from pathlib import Path

import matplotlib.pyplot as plt
from matplotlib.patches import FancyArrowPatch, Rectangle
from tensorflow.keras.models import load_model
from tensorflow.keras.utils import plot_model


MODEL_PATH = Path("../backend/model/music_classifier_model.h5")
OUTPUT_PATH = Path("model_architecture.png")


def shape_to_str(shape):
    if shape is None:
        return "unknown"
    if hasattr(shape, "as_list"):
        shape = shape.as_list()
    return "(" + ", ".join("None" if x is None else str(x) for x in shape) + ")"


def get_input_shape(layer_or_model):
    shape = getattr(layer_or_model, "input_shape", None)
    if shape is None and hasattr(layer_or_model, "input") and layer_or_model.input is not None:
        shape = layer_or_model.input.shape
    return shape_to_str(shape)


def get_output_shape(layer_or_model):
    shape = getattr(layer_or_model, "output_shape", None)
    if shape is None and hasattr(layer_or_model, "output") and layer_or_model.output is not None:
        shape = layer_or_model.output.shape
    return shape_to_str(shape)


def format_layer_title(layer, index, total_layers):
    layer_type = layer.__class__.__name__

    if index == total_layers - 1:
        return "Output Layer"
    if layer_type == "Bidirectional":
        return "Bidirectional\nLSTM"
    if layer_type == "Dropout":
        return f"Dropout\nLayer {index}"
    if layer_type == "Dense":
        return f"Dense\nLayer {index}"
    return layer_type.replace("_", " ")


def draw_reference_style_architecture(model, output_path):
    entries = [
        {
            "title": "Input",
            "input": get_input_shape(model),
            "output": get_input_shape(model),
        }
    ]

    total_layers = len(model.layers)
    for index, layer in enumerate(model.layers, start=1):
        entries.append(
            {
                "title": format_layer_title(layer, index, total_layers),
                "input": get_input_shape(layer),
                "output": get_output_shape(layer),
            }
        )

    left_w = 3.2
    middle_w = 1.9
    right_w = 3.7
    total_w = left_w + middle_w + right_w
    box_h = 1.5
    gap_h = 0.9
    margin_x = 0.4
    margin_top = 0.5
    margin_bottom = 0.4

    fig_h = margin_top + margin_bottom + len(entries) * box_h + (len(entries) - 1) * gap_h
    fig, ax = plt.subplots(figsize=(4.3, fig_h * 0.8))
    ax.set_xlim(0, total_w + margin_x * 2)
    ax.set_ylim(0, fig_h)
    ax.axis("off")
    ax.set_facecolor("white")

    y = fig_h - margin_top - box_h
    x0 = margin_x

    for i, entry in enumerate(entries):
        ax.add_patch(Rectangle((x0, y), left_w, box_h, fill=False, linewidth=1.1, edgecolor="black"))
        ax.add_patch(Rectangle((x0 + left_w, y), middle_w, box_h, fill=False, linewidth=1.1, edgecolor="black"))
        ax.add_patch(Rectangle((x0 + left_w + middle_w, y), right_w, box_h, fill=False, linewidth=1.1, edgecolor="black"))

        ax.plot([x0 + left_w, x0 + left_w + middle_w + right_w], [y + box_h / 2, y + box_h / 2], color="black", linewidth=1.0)

        ax.text(x0 + left_w / 2, y + box_h / 2, entry["title"], ha="center", va="center", fontsize=11, family="serif")
        ax.text(x0 + left_w + middle_w / 2, y + box_h * 0.74, "input:", ha="center", va="center", fontsize=10, family="serif", fontweight="bold")
        ax.text(x0 + left_w + middle_w / 2, y + box_h * 0.24, "output:", ha="center", va="center", fontsize=10, family="serif", fontweight="bold")
        ax.text(x0 + left_w + middle_w + right_w / 2, y + box_h * 0.74, entry["input"], ha="center", va="center", fontsize=10, family="serif")
        ax.text(x0 + left_w + middle_w + right_w / 2, y + box_h * 0.24, entry["output"], ha="center", va="center", fontsize=10, family="serif")

        if i < len(entries) - 1:
            arrow = FancyArrowPatch(
                (x0 + total_w / 2, y),
                (x0 + total_w / 2, y - gap_h + 0.15),
                arrowstyle="Simple,tail_width=0.4,head_width=6,head_length=8",
                color="black",
                linewidth=0.8,
            )
            ax.add_patch(arrow)

        y -= box_h + gap_h

    fig.tight_layout(pad=0.2)
    fig.savefig(output_path, dpi=220, bbox_inches="tight", facecolor="white")
    plt.close(fig)


model = load_model(MODEL_PATH)
model.summary()

try:
    plot_model(
        model,
        to_file=str(OUTPUT_PATH),
        show_shapes=True,
        show_layer_names=True,
        expand_nested=True,
        dpi=100,
    )
except Exception:
    draw_reference_style_architecture(model, OUTPUT_PATH)

if not OUTPUT_PATH.exists():
    draw_reference_style_architecture(model, OUTPUT_PATH)

print(f"Architecture image saved to: {OUTPUT_PATH.resolve()}")

Model: "sequential_3"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ bidirectional_3 (Bidirectional) │ (None, 128)            │        53,760 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_6 (Dropout)             │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_6 (Dense)                 │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_7 (Dropout)             │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_7 (Dense)                 │ (None, 2)              │           130 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 62,148 (242.77 KB)

 Trainable params: 62,146 (242.76 KB)

 Non-trainable params: 0 (0.00 B)

 Optimizer params: 2 (12.00 B)

You must install graphviz (see instructions at https://graphviz.gitlab.io/download/) for `plot_model` to work.
Architecture image saved to: model_architecture.png
